In [1]:
# Cell 1: Environment Setup
!pip install -q "numpy<2" "opencv-python==4.11.0.86" --force-reinstall
import os
print("Environment dependencies installed. Proceeding to main execution...")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 74.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.2 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have 

In [2]:
import cv2
import numpy as np
import os

# 1. Setup Output Directory
output = "results"
if not os.path.exists(output):
    os.makedirs(output)

# 2. Pewpare Test Image

image_path = "/content/IMG_5706.jpg"

img_raw = cv2.imread(image_path)
if img_raw is None:
    print(f"Error: Could not find '{image_path}'. Please upload image!!!")
else:
    # Crop top ~18% to foucs more on the architectural features
    h_raw, w_raw, _ = img_raw.shape
    img_cropped = img_raw[int(h_raw * 0.18):, :]
    cv2.imwrite(os.path.join(output, "base_cropped.jpg"), img_cropped)

    # Convert to grayscale for feature algorithms
    gray_base = cv2.cvtColor(img_cropped, cv2.COLOR_BGR2GRAY)
    h, w = gray_base.shape


    # 3. Create Transformed Variations

    # Transformation 1: 45-degree Rotation
    rot_matrix = cv2.getRotationMatrix2D((w // 2, h // 2), 45, 1.0)
    gray_rotated = cv2.warpAffine(gray_base, rot_matrix, (w, h))

    # Transformation 2: Scale Reduction (50%)
    gray_scaled = cv2.resize(gray_base, (0, 0), fx=0.5, fy=0.5)

    # Transformation 3: Illumination Attenuation (-60%)
    gray_dark = cv2.convertScaleAbs(gray_base, alpha=0.4, beta=-30)

    # Transformation 4: Gaussian Blur (9x9 Kernel)
    gray_blurred = cv2.GaussianBlur(gray_base, (9, 9), 0)

    test_cases = {
        "Rotation (45 deg)": gray_rotated,
        "Scale (0.5x)": gray_scaled,
        "Illumination (-60%)": gray_dark,
        "Gaussian Blur (9x9)": gray_blurred
    }

    # Save transformed images to output folder
    for name, trans_img in test_cases.items():
        clean_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("%", "")
        cv2.imwrite(os.path.join(output, f"transformed_{clean_name}.jpg"), trans_img)


    # 4. Initialize Feature Detectors & Descriptors

    # FAST Detector
    fast = cv2.FastFeatureDetector_create(threshold=25, nonmaxSuppression=True)

    # Standard ORB (Oriented FAST + Rotated BRIEF)
    orb = cv2.ORB_create(nfeatures=1000)

    # Unoriented ORB engine used to simulate standard BRIEF (fixed angle = 0)
    orb_unoriented = cv2.ORB_create(nfeatures=1000, WTA_K=2)

    # Brute-force matcher using Hamming distance for binary descriptors
    bf_matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    # 5. Helper Function to Simulate Standard BRIEF

    def compute_brief_simulated(detector, img, base_kps=None):
        """Detects FAST keypoints and computes unoriented binary BRIEF descriptors."""
        if base_kps is None:
            kps = detector.detect(img, None)
        else:
            kps = base_kps

        # Set orientation angle to 0 to simulate axis-aligned BRIEF tests
        for kp in kps:
            kp.angle = 0.0

        kps, des = orb_unoriented.compute(img, kps)
        return kps, des

    # 6. Extract Baseline Features on Original Image

    kp_fast_base = fast.detect(gray_base, None)
    kp_brief_base, des_brief_base = compute_brief_simulated(fast, gray_base, kp_fast_base)
    kp_orb_base, des_orb_base = orb.detectAndCompute(gray_base, None)

    print("=" * 65)
    print("Base Image Features Extracted:")
    print(f" - FAST keypoints detected : {len(kp_fast_base)}")
    print(f" - BRIEF descriptors computed: {len(kp_brief_base) if kp_brief_base is not None else 0}")
    print(f" - ORB keypoints detected   : {len(kp_orb_base)}")
    print("=" * 65)


    # 7. Benchmark Across Transformations

    print(f"{'Transformation':<22} | {'FAST KPs':<10} | {'BRIEF Matches':<13} | {'ORB Matches':<11}")
    print("-" * 65)

    for name, trans_img in test_cases.items():
        # 1. FAST Detection
        kp_fast_trans = fast.detect(trans_img, None)

        # 2. BRIEF Compute & Match (Unoriented)
        kp_brief_trans, des_brief_trans = compute_brief_simulated(fast, trans_img, kp_fast_trans)
        good_brief_matches = []
        if des_brief_base is not None and des_brief_trans is not None:
            raw_brief = bf_matcher.match(des_brief_base, des_brief_trans)
            good_brief_matches = [m for m in raw_brief if m.distance < 50]

        # 3. ORB Compute & Match
        kp_orb_trans, des_orb_trans = orb.detectAndCompute(trans_img, None)
        good_orb_matches = []
        if des_orb_base is not None and des_orb_trans is not None:
            raw_orb = bf_matcher.match(des_orb_base, des_orb_trans)
            good_orb_matches = [m for m in raw_orb if m.distance < 50]

        # Display quantitative comparison table row
        print(f"{name:<22} | {len(kp_fast_trans):<10} | {len(good_brief_matches):<13} | {len(good_orb_matches):<11}")

        # Draw and save ORB matching visual
        match_img = cv2.drawMatches(
            gray_base, kp_orb_base,
            trans_img, kp_orb_trans,
            good_orb_matches[:30], None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        clean_filename = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("%", "")
        save_path = os.path.join(output, f"orb_match_{clean_filename}.jpg")
        cv2.imwrite(save_path, match_img)

    print("=" * 65)
    print(f"Success! Output images and benchmark table generated in Colab's '{output}/' directory.")

Base Image Features Extracted:
 - FAST keypoints detected : 14498
 - BRIEF descriptors computed: 14235
 - ORB keypoints detected   : 1000
Transformation         | FAST KPs   | BRIEF Matches | ORB Matches
-----------------------------------------------------------------
Rotation (45 deg)      | 8693       | 892           | 692        
Scale (0.5x)           | 9743       | 1052          | 282        
Illumination (-60%)    | 1509       | 1122          | 104        
Gaussian Blur (9x9)    | 1865       | 1456          | 513        
Success! Output images and benchmark table generated in Colab's 'results/' directory.
